# Marketing Mix Modeling: Budget & ROI Optimization

In this project, I will build a **Machine Learning model to measure the impact of marketing investments on sales**. It involves **supervised learning (Regression)**, where the target is the **sales volume ($y$)** based on expenditures across various media channels **($X$)**.

I will use the following pipeline based on the **CRISP-DM framework**:

1. **Define the business problem.**
2. **Collect the data and get a general overview of it.**
3. **Split the data into train and test sets.**
4. **Explore the data (Exploratory Data Analysis).**
5. **Feature engineering and preprocessing (Adstock and Saturation).**
6. **Model training and hyperparameter tuning.**
7. **Model testing and evaluation.**
8. **Interpret results (ROI and Contribution).**
9. **Optimization Plan.**

---

In this notebook, I will perform **Exploratory Data Analysis (EDA)**, covering **steps 1 to 4 of the pipeline above**.

The main objective here is to **uncover insights into historical sales behavior** relative to the **investment channels used**, enabling a better understanding of how different media contributions impact overall performance.


# 1. Business problem

The central challenge is the **lack of visibility regarding the individual contribution of each sales channel** to the overall results. Without clear statistical modeling, investments are often distributed based on intuition, leading to **financial inefficiency** and **underutilization of channels with high scaling potential**.

## 1.1 What is the context?

In order to **optimize the marketing mix** and **maximize profitability**, three essential **Key Performance Indicators (KPIs)** are considered:

- **ROAS (Return on Ad Spend):** Measures the revenue generated for each monetary unit invested per channel. Higher ROAS reflects **efficient media spend**.

- **Sales Lift (Incrementality):** Represents the sales volume attributed exclusively to marketing efforts, isolating **organic growth** and **baseline sales**.

- **Marginal Acquisition Cost:** Measures the cost to generate the next sale in a specific channel, identifying **media saturation points**.

### Sales Channels Analyzed:

- **Digital:** Google Search (SEM), Social Media (Meta), and YouTube.

- **Offline:** TV (National/Regional) and Radio.

- **Support:** Email Marketing and Push Notifications.

## Which are the project objectives?

1. **Identify** the factors (channels) most associated with **revenue growth**.

2. **Construct** a model capable of accurately predicting **sales based on media plan changes**.

3. **Offer** action plans to **reallocate budget** from saturated channels to high-potential ones.

## Why the Marginal Contribution approach?

When deploying the model, the primary objective is to generate **contribution scores for each channel**. This is more valuable for business decision-making than simple total predictions, as it enables the company to understand **elasticity**.

This information allows for concentrating efforts on the **"sweet spot" of investment**, ordering channels from the **highest marginal return to the lowest**.

## Which are the benefits?

- **Cost Savings** by cutting spend on saturated media.

- **Improved Resource Allocation** based on **data-driven ROI**.

- **Enhanced Financial Predictability** for future sales cycles.

- **Targeted Budgeting** to protect **profit margins**.

In [2]:
# CONFIGURAÇÃO DE AMBIENTE — MARKETING MIX MODELING

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import sys
import os

# Garantindo acesso aos módulos src/
sys.path.append(os.path.abspath(os.path.join('../')))

# Desativar avisos
import warnings
warnings.filterwarnings('ignore')

### Configurações Visuais (Estilo Técnico Corporativo)
%matplotlib inline
mpl.style.use('ggplot') 

mpl.rcParams['axes.facecolor']      = 'white'
mpl.rcParams['axes.linewidth']      = 1
mpl.rcParams['xtick.color']         = 'black'
mpl.rcParams['ytick.color']         = 'black'
mpl.rcParams['grid.color']          = 'lightgray'
mpl.rcParams['figure.dpi']          = 150
mpl.rcParams['axes.grid']           = True
mpl.rcParams['font.size']           = 12

# Paleta Customizada: Foco em Contraste e Leitura Profissional
color_palette = ['#023047', '#e85d04', '#0077b6', '#ff8200', '#0096c7', '#ff9c33']
sns.set_palette(sns.color_palette(color_palette))

print("✅ Setup Visual e Estratégico do Business Understanding concluído.")
sns.color_palette(color_palette)

✅ Setup Visual e Estratégico do Business Understanding concluído.


[(0.00784313725490196, 0.18823529411764706, 0.2784313725490196),
 (0.9098039215686274, 0.36470588235294116, 0.01568627450980392),
 (0.0, 0.4666666666666667, 0.7137254901960784),
 (1.0, 0.5098039215686274, 0.0),
 (0.0, 0.5882352941176471, 0.7803921568627451),
 (1.0, 0.611764705882353, 0.2)]

In [1]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

from src.components.data_ingestion import DataIngestion

ingestao = DataIngestion()
df_mmm = ingestao.initiate_data_ingestion()
df_mmm.head()

--- DEBUG DE TAMANHOS DAS COLUNAS ---
Coluna: Data | Tamanho: 155
Coluna: Vendas | Tamanho: 156
Coluna: Investimento_TV | Tamanho: 156
Coluna: Investimento_Google_Ads | Tamanho: 156
Coluna: Investimento_Facebook_Ads | Tamanho: 156
Coluna: Fator_Feriado | Tamanho: 156
Coluna: Fator_Sazonalidade | Tamanho: 156
-------------------------------------


,Vendas,Investimento_TV,Investimento_Google_Ads,Investimento_Facebook_Ads,Fator_Feriado,Fator_Sazonalidade
Data,,,,,,
2023-04-10,63.204967,17.777792,20.552446,19.720831,6.230423,4.976091
2023-04-17,67.406395,17.394508,20.412626,20.925434,3.964424,4.463735
2023-04-24,64.561142,19.735489,16.875376,21.928518,3.937733,7.598067
2023-05-01,63.577545,19.313116,20.446329,20.642603,5.775819,4.731113
2023-05-08,64.768959,18.865713,20.808935,21.666994,5.862508,5.124802
